In [ ]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable


In [ ]:
catalog_name = "projectecommerce"

In [ ]:
days_cutoff = 30
source_table_name = "gld_fact_order_items"
table_name = "gld_fact_daily_orders_summary"

# Get max transaction_date safely
max_date_row = spark.sql(f"""
    SELECT MAX(transaction_date) AS max_date 
    FROM {catalog_name}.gold.{source_table_name}
""").collect()[0]

max_date = max_date_row['max_date']
print(max_date)

# Build WHERE clause safely
if max_date is None:
    where_clause = "1=1"
elif spark.catalog.tableExists(f"{catalog_name}.gold.{table_name}"):
    where_clause = f"transaction_date >= date_sub(date('{max_date}'), {days_cutoff})"
else:
    where_clause = "1=1"

# Aggregation (removed ORDER BY)
summary_query = f"""
SELECT
    date_id,
    unit_price_currency AS currency,
    SUM(quantity) AS total_quantity,
    SUM(gross_amount) AS total_gross_amount,
    SUM(discount_amount) AS total_discount_amount,
    SUM(tax_amount) AS total_tax_amount,
    SUM(net_amount) AS total_amount
FROM {catalog_name}.gold.{source_table_name}
WHERE {where_clause}
GROUP BY date_id, currency
"""

summary_df = spark.sql(summary_query)

# Debug
summary_df.show(7)

summary_df.select(
    F.min("date_id").alias("min_date"),
    F.max("date_id").alias("max_date")
).show()

target_table = f"{catalog_name}.gold.{table_name}"

# Initial load
if not spark.catalog.tableExists(target_table):
    summary_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

    spark.sql(f"""
        ALTER TABLE {target_table} 
        SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """)
else:
    delta_table = DeltaTable.forName(spark, target_table)

    delta_table.alias("gold_table").merge(
        summary_df.alias("data_snapshot"),
        "gold_table.date_id = data_snapshot.date_id AND gold_table.currency = data_snapshot.currency"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()